# NDgpu — Phase 0: device diffusion-solve bake-off (Colab)

**Go/no-go for porting CMFD to the GPU.** CMFD's cost is a diffusion solve
reused many times per outer (`_cmfd_power` calls `facs[g](q)` with the same
matrix, many right-hand sides). On the host, `scipy.factorized` amortises one
O(N^1.5) sparse LU over all those back-solves. On the GPU we lose that reuse, so
the question is: **is there a device solve whose "one setup + K back-solves"
total grows slower than host LU as the mesh refines?** If not, CMFD stays on the
host and the GPU advantage keeps sliding toward 1x with refinement.

We benchmark, on the **real** tri-diffusion operator (`_dsa_matrix`, same
sparsity/conditioning as the CMFD drift matrix) at several mesh sizes:

| strategy | reuse | expected scaling |
|---|---|---|
| host `scipy.factorized` (baseline) | LU once + K back-solves | ~O(N^1.5) setup |
| device Jacobi-CG (`linalg.pcg`) | none (K iterative solves) | ~O(N^1.5), iters ↑√N |
| device Neumann-CG (`neumann_preconditioner`) | none | ~O(N^1.5), smaller const |
| cupy sparse direct (`cupyx…spsolve`) | if available | ? |
| **multigrid** — CPU pyamg + **GPU V-cycle** | hierarchy once + K V-cycles | **O(N), iters flat** |

The decisive plot is **iterations vs N**: Jacobi-CG climbs ~√N; multigrid stays
flat. Flat iterations × O(N) per V-cycle = O(N) — the only strategy that makes
the GPU advantage *grow* with refinement.

(Cross sections are illustrative placeholders, not predictive.)

In [ ]:
import os
try:                                        # Colab: upload dist/ndgpu-src.zip
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    get_ipython().run_line_magic("pip", f"install -q {zip_name}")
    try:
        import cupy
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q cupy-cuda12x")
    get_ipython().run_line_magic("pip", "install -q pyamg")
    get_ipython().system("nvidia-smi -L")
except ImportError:                         # local run: ndgpu already importable
    get_ipython().run_line_magic("pip", "install -q pyamg") if False else None

In [ ]:
import time
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import pyamg

from ndgpu.benchmarks import build_hpmr2d
from ndgpu.tri_sn import TriSNTransportSolver
from ndgpu.linalg import pcg, neumann_preconditioner

try:
    import cupy
    import cupyx.scipy.sparse as csp
    import cupyx.scipy.sparse.linalg as cspla
    HAVE_GPU = cupy.cuda.runtime.getDeviceCount() > 0
except Exception:
    HAVE_GPU = False
print("GPU available:", HAVE_GPU)

QUICK = bool(os.environ.get("NDGPU_QUICK"))
REFINES = [4, 6] if QUICK else [4, 6, 8, 10, 12]
K = 3 if QUICK else 30          # back-solves per setup (~ _cmfd_power call count)
RTOL = 1e-6                     # CMFD-relevant inner tolerance


def real_matrix(refine):
    """The real group-0 tri diffusion operator at this refinement (host CSR)."""
    p = build_hpmr2d(refine=refine, drum_angle_deg=0.0, absorber="polar")
    s = TriSNTransportSolver(p.grid, p.materials, p.material_map, active=p.active,
                             bc="vacuum", scheme="scb", engine="levels",
                             device="cpu", n_polar=2, n_azi=8,
                             mix_material=p.mix_material, mix_weight=p.mix_weight)
    return s._dsa_matrix(0).tocsr()


def xsync(xp):
    if xp is not np:
        xp.cuda.Stream.null.synchronize()


def xp_csr(A_host, xp):
    return A_host if xp is np else csp.csr_matrix(A_host)


def rhs_batch(N, xp, seed=0):
    B = np.random.default_rng(seed).standard_normal((K, N))
    return B if xp is np else xp.asarray(B)

In [ ]:
# ---- multigrid V-cycle: pyamg builds the hierarchy (CPU), the apply runs on
# ---- the chosen backend (numpy or cupy) so it is testable on CPU and portable.
def build_mg(A_host, xp, presmooth=2, postsmooth=2):
    ml = pyamg.smoothed_aggregation_solver(A_host, max_coarse=400)
    levels = []
    for L in ml.levels[:-1]:
        A = L.A.tocsr()
        d = A.diagonal()
        levels.append(dict(A=xp_csr(A, xp),
                           Dinv=xp.asarray(1.0 / d) if xp is not np else 1.0 / d,
                           P=xp_csr(L.P.tocsr(), xp), R=xp_csr(L.R.tocsr(), xp)))
    Ac = ml.levels[-1].A.tocsc()             # coarsest: exact host LU (tiny)
    coarse_lu = spla.factorized(Ac)
    omega = 0.7                              # damped-Jacobi smoother

    def vcycle(b, i=0):
        if i == len(levels):
            xc = coarse_lu(np.asarray(b) if xp is np else cupy.asnumpy(b))
            return xc if xp is np else xp.asarray(xc)
        lv = levels[i]
        x = xp.zeros_like(b)
        for _ in range(presmooth):
            x = x + omega * lv["Dinv"] * (b - lv["A"] @ x)
        r = b - lv["A"] @ x
        ec = vcycle(lv["R"] @ r, i + 1)
        x = x + lv["P"] @ ec
        for _ in range(postsmooth):
            x = x + omega * lv["Dinv"] * (b - lv["A"] @ x)
        return x

    n_levels = len(levels) + 1
    return vcycle, n_levels


def cg_iters(apply_A, b, xp, precond, rtol):
    """CG with a custom preconditioner; returns (x, n_iters)."""
    return pcg(apply_A, b, xp.zeros_like(b), None, xp, rtol=rtol,
               maxiter=5000, precond=precond, raise_on_fail=False)

In [ ]:
def strat_host_lu(A_host, B_host):
    t = time.perf_counter(); solve = spla.factorized(A_host.tocsc())
    setup = time.perf_counter() - t
    t = time.perf_counter()
    for b in B_host:
        solve(b)
    return dict(name="host-LU", setup=setup, ksolve=time.perf_counter() - t, iters=0)


def strat_iterative(name, precond_of, A_host, xp):
    Ad = xp_csr(A_host, xp)
    inv_diag = xp.asarray(1.0 / A_host.diagonal()) if xp is not np \
        else 1.0 / A_host.diagonal()
    apply_A = lambda x: Ad @ x
    t = time.perf_counter()
    precond = precond_of(apply_A, inv_diag)
    xsync(xp); setup = time.perf_counter() - t
    B = rhs_batch(A_host.shape[0], xp)
    xsync(xp); t = time.perf_counter(); its = []
    for j in range(K):
        _, it = cg_iters(apply_A, B[j], xp, precond, RTOL)
        its.append(it)
    xsync(xp)
    return dict(name=name, setup=setup, ksolve=time.perf_counter() - t,
                iters=float(np.mean(its)))


def strat_mg(A_host, xp):
    B = rhs_batch(A_host.shape[0], xp)
    Ad = xp_csr(A_host, xp)
    apply_A = lambda x: Ad @ x
    t = time.perf_counter()
    vcycle, nlev = build_mg(A_host, xp)
    xsync(xp); setup = time.perf_counter() - t
    xsync(xp); t = time.perf_counter(); its = []
    for j in range(K):
        _, it = cg_iters(apply_A, B[j], xp, lambda r: vcycle(r), RTOL)
        its.append(it)
    xsync(xp)
    return dict(name="MG", nlev=nlev, setup=setup,
                ksolve=time.perf_counter() - t, iters=float(np.mean(its)))


def strat_cupy_direct(A_host, xp):
    if xp is np:
        return None
    Ad = csp.csr_matrix(A_host)
    B = rhs_batch(A_host.shape[0], xp)
    xsync(xp); t = time.perf_counter()
    for j in range(K):
        cspla.spsolve(Ad, B[j])              # no factor reuse in cupy
    xsync(xp)
    return dict(name="cupy-direct", setup=0.0,
                ksolve=time.perf_counter() - t, iters=0)

In [ ]:
def run_bake_off(xp, tag):
    print(f"\n==== {tag}: setup + {K} back-solves, real tri-diffusion operator ====")
    print(f"{'strategy':>12} {'N':>7} {'setup[s]':>9} {'Ksolve[s]':>10} "
          f"{'per_solve[ms]':>13} {'avg_iters':>9}")
    data = {}
    for refine in REFINES:
        A = real_matrix(refine)
        N = A.shape[0]
        results = []
        if xp is np:
            results.append(strat_host_lu(A, rhs_batch(N, np)))
        results.append(strat_iterative(
            "jacobi-CG", lambda aA, invd: (lambda r: invd * r), A, xp))
        results.append(strat_iterative(
            "neumann-CG",
            lambda aA, invd: neumann_preconditioner(aA, invd, 6), A, xp))
        results.append(strat_mg(A, xp))
        cd = strat_cupy_direct(A, xp)
        if cd:
            results.append(cd)
        for r in results:
            r["N"] = N
            data.setdefault(r["name"], []).append(r)
            print(f"{r['name']:>12} {N:>7} {r['setup']:>9.3f} {r['ksolve']:>10.3f} "
                  f"{1e3*r['ksolve']/K:>13.2f} {r['iters']:>9.1f}")
    return data

cpu_data = run_bake_off(np, "CPU")
gpu_data = run_bake_off(cupy, "GPU") if HAVE_GPU else {}

In [ ]:
import matplotlib.pyplot as plt

def plot(ax, data, field, title, ylabel, logy=True):
    for name, rows in data.items():
        Ns = [r["N"] for r in rows]
        ys = [r[field] for r in rows]
        ax.plot(Ns, ys, "o-", label=name)
    ax.set(xlabel="N (cells)", ylabel=ylabel, title=title, xscale="log")
    if logy:
        ax.set_yscale("log")
    ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=8)

src = gpu_data if HAVE_GPU else cpu_data
lbl = "GPU" if HAVE_GPU else "CPU"
fig, ax = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
plot(ax[0], src, "ksolve", f"{lbl}: {K} back-solves (total)", "wall [s]")
plot(ax[1], {k: v for k, v in src.items() if v[0]["iters"] > 0},
     "iters", f"{lbl}: iterations vs N (flat = O(N))", "avg iters", logy=False)
plt.show()

# reference slopes on the total-solve plot
print("A power-law fit  ksolve ~ N^p  (p~1 is O(N), p~1.5 is sparse-LU-like):")
for name, rows in src.items():
    Ns = np.array([r["N"] for r in rows], float)
    ys = np.array([r["ksolve"] for r in rows], float)
    if len(Ns) >= 2 and (ys > 0).all():
        p = np.polyfit(np.log(Ns), np.log(ys), 1)[0]
        print(f"  {name:>12}: p = {p:.2f}")

## Reading the results — the go/no-go

* **iterations vs N** (right plot) is the crux. `jacobi-CG` should climb ~√N;
  `MG` should stay ~flat (7–13 here on CPU). Flat iterations are the O(N)
  signature — the only way a device solve beats host LU asymptotically.
* **power-law exponent `p`** on the total back-solve time: host-LU and the
  unpreconditioned iterative solvers trend toward `p ≈ 1.4–1.5`; a working GPU
  multigrid should trend toward `p ≈ 1.0`.
* **GO** if GPU `MG` back-solve time grows near-linearly and, at the largest N,
  beats host-LU by a margin that widens with N → CMFD on the device will make
  the full-solver speed-up grow with refinement (Phases 1–4).
* **NO-GO** if even multigrid can't beat host-LU on the device across the tested
  N — then CMFD stays on the host and we accept the ceiling.

Caveats: this is the **symmetric** diffusion operator; the CMFD drift matrix
adds a small non-symmetry (handled by BiCGStab / a non-symmetric MG cycle in
Phase 2 — same sparsity and conditioning, so the scaling verdict carries).
`cupy-direct` (`spsolve`) has no factor reuse, so it re-factorises every
back-solve — expect it to lose; it's here only to confirm that.